<a href="https://colab.research.google.com/github/sebr22/sebr/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sebr22/sebr/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis + time window**

The underlying warehouse stores one row per content page, per client, per day. For this project, I will use daily records from March 2026 and aggregate them into a monthly summary for each content page before building my model. The model will then rank pages according to their potential content-refresh opportunity, using only information available at the time the ranking is made. I deliberately exclude any information from future dates or any columns derived from the target label to avoid data leakage and ensure the ranking reflects the information that would have been available when the decision was made.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

The features I plan to use are `gsc_impressions`, `gsc_clicks`, `gsc_avg_position`, `ga4_sessions`, and `ga4_engaged_sessions`. These are all available at the time the content review decision is made and provide information about a page's search visibility, search performance, and user engagement.

The label (or proxy) will be defined later in the project based on the content-refresh ranking task. For this data contract, I am identifying only the fields that are available before the prediction is made.

The context fields are `content_hash_id`, `client_hash_id`, and `report_date`. These identify the page, client, and observation date but are not intended to be predictive features.

I will exclude any future information, such as performance from later dates or any columns derived from the target label. I will also exclude existing decision scores or product-generated flags because they would leak information about the desired outcome and lead to overly optimistic model performance.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

I verified my data contract using SQL queries against the fact_content_daily_performance table.
First, I confirmed the observation window. The table contains 78,835,655 rows spanning 27 January 2025 to 30 June 2026, confirming that March 2026 is a valid mid-panel month for developing my features while leaving the final month available for evaluation.
Next, I verified the grain of the data. Filtering to March 2026 returned 9,841,378 rows, and there were also 9,841,378 unique combinations of client_hash_id, content_hash_id, and report_date. This confirms that the table contains one row per content page, per client, per day.
Finally, I checked data availability. In March 2026, 3,611,061 rows have Google Search Console data available, 413,966 have Google Analytics 4 data available, and 364,347 have both data sources available. This shows that not every row contains all data sources, so analyses using GA4 metrics should filter to rows where the relevant availability flags are TRUE.

```SQL
SELECT
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date,
    COUNT(*) AS total_rows
FROM fact_content_daily_performance;
```
```SQL
SELECT
    COUNT(*) AS rows_march_2026
FROM fact_content_daily_performance
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01';
```
```SQL
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
          AND ga4_data_available IS TRUE
    ) AS both_available
FROM fact_content_daily_performance
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01';
```
```SQL
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT client_hash_id || '-' || content_hash_id || '-' || CAST(report_date AS VARCHAR)) AS unique_keys
FROM fact_content_daily_performance
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01';
```

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This dataset has several limitations. It cannot show whether a content refresh directly caused a page's performance to improve, so it cannot support causal conclusions. The historical coverage may also be unbalanced, with some pages having longer observation periods than others, and the earliest rows may contain only Google Search Console data before Analytics data became available. In addition, care must be taken to avoid overlapping observation and outcome windows, as this could introduce data leakage and lead to overly optimistic model performance. As a result, the model should be used to support prioritisation decisions rather than to predict future performance with certainty.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.